## What We Will Build In Colab

A fully working persistent multi-user AI chat system:

User → API Function → SQLite DB → HuggingFace Model → Response

Features we will get:

-  Persistent chat database
- Multiple users supported
- Conversation memory (context)
- Role-based chatbot
- No OpenAI key required

## STEP 0 — Open Google Colab

Create new notebook → name it:

AI_Chat_System.ipynb

## STEP 1 — Install Required Libraries

Run this cell
```
!pip install transformers accelerate sentencepiece
```

Why we install:
| Library       | Purpose             |
| ------------- | ------------------- |
| transformers  | Load free AI models |
| accelerate    | Faster inference    |
| sentencepiece | Tokenizer support   |


In [1]:
!pip install transformers accelerate sentencepiece

## STEP 2 — Load Open-Source AI Model (FREE)

We will use a lightweight conversational model.
```
from transformers import pipeline

chatbot = pipeline(
    "text-generation",
    model="distilgpt2"
)

print("Model loaded successfully")
```
Why distilgpt2?

Small → Fast → Runs in Colab CPU.

In [2]:
from transformers import pipeline

chatbot = pipeline(
    "text-generation",
    model="distilgpt2"
)

print("Model loaded successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model loaded successfully


## STEP 3 — Create SQLite Database (Persistent Memory)

Now we build your chat database.
```
import sqlite3

def init_db():
    conn = sqlite3.connect("chat.db")
    cursor = conn.cursor()
    
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS chats (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_id TEXT,
        message TEXT,
        response TEXT
    )
    """)
    
    conn.commit()
    conn.close()

init_db()
print("Database created")
```

 SQLite creates a real file called chat.db inside Colab.

In [3]:
import sqlite3

def init_db():
    conn = sqlite3.connect("chat.db")
    cursor = conn.cursor()

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS chats (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_id TEXT,
        message TEXT,
        response TEXT
    )
    """)

    conn.commit()
    conn.close()

init_db()
print("Database created")

Database created


## STEP 4 — Function to Save Chat
```
def save_chat(user_id, message, response):
    conn = sqlite3.connect("chat.db")
    cursor = conn.cursor()
    
    cursor.execute(
        "INSERT INTO chats (user_id, message, response) VALUES (?, ?, ?)",
        (user_id, message, response)
    )
    
    conn.commit()
    conn.close()
```

In [4]:
def save_chat(user_id, message, response):
    conn = sqlite3.connect("chat.db")
    cursor = conn.cursor()

    cursor.execute(
        "INSERT INTO chats (user_id, message, response) VALUES (?, ?, ?)",
        (user_id, message, response)
    )

    conn.commit()
    conn.close()

## STEP 5 — Function to Load Chat History

This gives chatbot memory.
```
def get_history(user_id):
    conn = sqlite3.connect("chat.db")
    cursor = conn.cursor()
    
    cursor.execute(
        "SELECT message, response FROM chats WHERE user_id=?",
        (user_id,)
    )
    
    rows = cursor.fetchall()
    conn.close()
    
    return rows
```

In [5]:
def get_history(user_id):
    conn = sqlite3.connect("chat.db")
    cursor = conn.cursor()

    cursor.execute(
        "SELECT message, response FROM chats WHERE user_id=?",
        (user_id,)
    )

    rows = cursor.fetchall()
    conn.close()

    return rows

## STEP 6 — Create Role-Based Personalities

Industry systems always use system prompts.
```
roles = {
    "teacher": "You are a helpful AI teacher who explains clearly.",
    "interview": "You are a job interview coach giving professional answers.",
    "strict": "Answer in only 2 short lines."
}
```

In [6]:
roles = {
    "teacher": "You are a helpful AI teacher who explains clearly.",
    "interview": "You are a job interview coach giving professional answers.",
    "strict": "Answer in only 2 short lines."
}

## STEP 7 — Build the MAIN Chat Function (API Simulation)

This replaces Flask API in Colab.
```
def chat_api(user_id, role, message):
    
    # 1. Load past conversation
    history = get_history(user_id)
    
    # 2. Build prompt with memory
    prompt = roles[role] + "\n"
    
    for msg, res in history:
        prompt += f"User: {msg}\nBot: {res}\n"
    
    prompt += f"User: {message}\nBot:"
    
    # 3. Generate response
    output = chatbot(prompt, max_length=200, do_sample=True)
    reply = output[0]["generated_text"].split("Bot:")[-1]
    
    # 4. Save to database
    save_chat(user_id, message, reply)
    
    return reply
```
This function simulates a real backend API.

In [7]:
def chat_api(user_id, role, message):

    # 1. Load past conversation
    history = get_history(user_id)

    # 2. Build prompt with memory
    prompt = roles[role] + "\n"

    for msg, res in history:
        prompt += f"User: {msg}\nBot: {res}\n"

    prompt += f"User: {message}\nBot:"

    # 3. Generate response
    output = chatbot(prompt, max_length=200, do_sample=True)
    reply = output[0]["generated_text"].split("Bot:")[-1]

    # 4. Save to database
    save_chat(user_id, message, reply)

    return reply

## STEP 8 — Create Multi-User Sessions

Now we simulate different users.
```
import uuid

# Create two users
user1 = str(uuid.uuid4())
user2 = str(uuid.uuid4())

print("User1:", user1)
print("User2:", user2)
```

Each user has separate memory

In [8]:
import uuid

# Create two users
user1 = str(uuid.uuid4())
user2 = str(uuid.uuid4())

print("User1:", user1)
print("User2:", user2)

User1: 5e6e1615-c0f4-4305-a620-61edee919029
User2: 744f293e-bfef-4ed2-a681-0c9ed3760ffb


## STEP 9 — Chat With User 1

```
chat_api(user1, "teacher", "What is machine learning?")
```
Run again:
```
chat_api(user1, "teacher", "Give an example")
```

Notice: It remembers the first question!


In [9]:
chat_api(user1, "teacher", "What is machine learning?")

Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


' It is the most complex algorithm. It has many types of algorithms. It has many types of algorithms. It has many types of'

In [10]:
chat_api(user1, "teacher", "Give an example")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'                                                                                                                                                                '

### STEP 10 — Chat With User 2 (Separate Memory)

```
chat_api(user2, "interview", "Tell me about yourself")
```

User2 has fresh conversation.

This = Multi-user system

In [11]:
chat_api(user2, "interview", "Tell me about yourself")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


" I'm a great recruiter. I have my own company. I have my own company. I have my own company. I have my own company. I have my own company. I have my own company. I have my own company. I have my own company. I have my own company. I have my own company. I have my own company. I have my own company. I have my own company."

## STEP 11 — See Stored Database (Proof)

```
conn = sqlite3.connect("chat.db")
cursor = conn.cursor()

cursor.execute("SELECT * FROM chats")
rows = cursor.fetchall()

for row in rows:
    print(row)
```
We will see ALL chats saved permanently

In [12]:
conn = sqlite3.connect("chat.db")
cursor = conn.cursor()

cursor.execute("SELECT * FROM chats")
rows = cursor.fetchall()

for row in rows:
    print(row)

(1, '5e6e1615-c0f4-4305-a620-61edee919029', 'What is machine learning?', ' It is the most complex algorithm. It has many types of algorithms. It has many types of algorithms. It has many types of')
(2, '5e6e1615-c0f4-4305-a620-61edee919029', 'Give an example', '                                                                                                                                                                ')
(3, '744f293e-bfef-4ed2-a681-0c9ed3760ffb', 'Tell me about yourself', " I'm a great recruiter. I have my own company. I have my own company. I have my own company. I have my own company. I have my own company. I have my own company. I have my own company. I have my own company. I have my own company. I have my own company. I have my own company. I have my own company. I have my own company.")


## What I Just Built (Industry Level)

We now understand real AI backend architecture:

| Component   | What you built     |
| ----------- | ------------------ |
| AI Model    | HuggingFace GPT    |
| Backend API | chat_api()         |
| Database    | SQLite             |
| Memory      | Chat history       |
| Multi-user  | UUID sessions      |
| Role system | Prompt engineering |

This is the foundation of ChatGPT-style apps.